In [ ]:
from pathlib import Path
import os
if Path.cwd().name == "examples" and (Path.cwd().parent / "pyproject.toml").exists():
    os.chdir(Path.cwd().parent)
print("Working directory:", Path.cwd())


# MPRALegNet for cell-selective enhancer design

[Notebook](multiple-models.ipynb) · [Installation](../docs/installation.md)

Combine the pretrained K562, HepG2 and WTC11 MPRALegNet predictors. Reference predictions put the three outputs on standardized scales before optimization.

## Load the predictors and set the objective



In [ ]:
import numpy as np
import torch
from bioseq_models import load_model
from sequence_optimization import CombinedScorer, MultiTaskObjective, generate_designs, random_seed_sequences

device = "cuda" if torch.cuda.is_available() else "cpu"
cells = ["K562", "HepG2", "WTC11"]
model = CombinedScorer(
    [load_model(f"agarwal_{cell.lower()}", device=device) for cell in cells],
    names=cells,
)
reference = model.score(random_seed_sequences(230, 32, seed=43)).scores
objective = MultiTaskObjective(
    high=["K562"], low=["HepG2", "WTC11"], aggregation="mean",
    centers=reference.mean(axis=0), scales=np.maximum(reference.std(axis=0), 1e-8),
)
seeds = random_seed_sequences(230, 4, seed=42)




## Optimize and inspect the candidates



In [ ]:
designs = generate_designs(
    model, length=230, initial=seeds, objective=objective,
    algorithms=["simulated_annealing"], num_candidates=4, seed=42, parallel=False,
)
designs[["activity_K562", "activity_HepG2", "activity_WTC11", "score"]]
